# Knee v6: ConvNeXt-Tiny, 20 epochs, 5-fold (`TAG = v6cnxT20`)
Fold 0 of this recipe was already trained in the screen notebook (`screen_cnxT_e20/`). If you attach that notebook's output, this run reuses fold 0 and trains folds 1-4 (GPU 0: folds 1, 2 | GPU 1: folds 3, 4, about 5.2 h wall, estimate from ~7.8 min/epoch with two jobs). If it is not attached, folds 0-2 run on GPU 0 and 3-4 on GPU 1 (about 7.8 h).
Recipe is identical to the screen: `convnext_tiny.in12k_ft_in1k`, head tokens, K=1, 160 px, 20 epochs, bs 4 x accum 2, train_slices 16, seed 42.
Outputs: `models_v6cnxT20/fold{k}_v6cnxT20.pt` (unique names), `oof_v6cnxT20.csv`.
Inputs: `knee-mri-pipeline` (patched v6_3), caching output (cache + `labels_soft.csv`), baseline training output (`oof_all.csv`), ConvNeXt 5-fold outputs (`oof_v6convnext.csv`, `oof_v6convnext2.csv`), and the screen notebook output (for fold 0). GPU T4 x2, internet ON, Save & Run All.

In [1]:
import os, sys, json, subprocess, time, threading, shutil
from pathlib import Path
import numpy as np, pandas as pd

TAG = "v6cnxT20"
BACKBONE, EPOCHS = "convnext_tiny.in12k_ft_in1k", 20
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
for n, p in dict(CODE=CODE, CACHE=CACHE, LABELS=LABELS).items(): assert p.exists(), f"{n} not found: {p}"
assert (CODE / "train.py").exists() and (CODE / "geom.py").exists(), "dataset does not hold knee_pipeline_v6"

# fold 0 from the screen notebook, if attached
SKIP = {"cache", "train_series", "test_series", "train", "test", "__pycache__", ".git"}
src0 = []
for d, dirs, fs in os.walk(ROOT):
    dirs[:] = [x for x in dirs if x not in SKIP]
    if Path(d).name == "screen_cnxT_e20" and "fold0.pt" in fs and "oof.csv" in fs: src0.append(Path(d))
assert len(src0) <= 1, f"several screen_cnxT_e20 folders: {src0}"
SCREEN = src0[0] if src0 else None
print("fold-0 source:", SCREEN or "NOT attached -> fold 0 will be retrained")
JOBS = {0: (f"{TAG}_g0", ["1", "2"] if SCREEN else ["0", "1", "2"]), 1: (f"{TAG}_g1", ["3", "4"])}
print(JOBS)
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})"); assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"

fold-0 source: /kaggle/input/notebooks/theokorir/knee-fold0-backbone-screen/screen_cnxT_e20
{0: ('v6cnxT20_g0', ['1', '2']), 1: ('v6cnxT20_g1', ['3', '4'])}
== tests/test_geom.py (exit 0)
== tests/smoke_v6.py (exit 0)


In [2]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--pretrained", "--workers", "2", "--train_slices", "16",
             "--head", "tokens", "--K", "1", "--seed", "42", "--backbone", BACKBONE, "--epochs", str(EPOCHS), "--bs", "4", "--accum", "2"]
LOGS = {}
def run(gpu, name, folds):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + ["--only_folds"] + folds + ["--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    with open(LOGS[name], "w") as fh:
        rc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE)).wait()
    print(f"[{name}] exit code {rc}", flush=True)
threads = [threading.Thread(target=run, args=(g, n, f)) for g, (n, f) in JOBS.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(900)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if " ep " in l] if f.exists() else []
        last[n] = ls[-1] if ls else "(none yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished")

[v6cnxT20_g0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --pretrained --workers 2 --train_slices 16 --head tokens --K 1 --seed 42 --backbone convnext_tiny.in12k_ft_in1k --epochs 20 --bs 4 --accum 2 --only_folds 1 2 --out /kaggle/working/v6cnxT20_g0
[v6cnxT20_g1] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --pretrained --workers 2 --train_slices 16 --head tokens --K 1 --seed 42 --backbone convnext_tiny.in12k_ft_in1k --epochs 20 --bs 4 --accum 2 --only_folds 3 4 --out /kaggle/working/v6cnxT20_g1
[0.25 h] {
 "v6cnxT20_g0": "[f1] ep 1/20 loss 0.8067 | val mac

In [3]:
# ---- collect: unique model names + merged OOF (fold 0 from the screen if reused) ----
M = W / f"models_{TAG}"; M.mkdir(exist_ok=True)
sources = {k: (W / name / f"fold{k}.pt") for name, folds in JOBS.values() for k in map(int, folds)}
if SCREEN: sources[0] = SCREEN / "fold0.pt"
for k in range(5):
    assert sources[k].exists(), f"missing {sources[k]}: a job failed, do not use this output"
    shutil.copy(sources[k], M / f"fold{k}_{TAG}.pt")
assert len(list(M.glob("*.pt"))) == 5; print(sorted(p.name for p in M.glob("*.pt")))
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
parts = [pd.read_csv(W / JOBS[0][0] / "oof.csv"), pd.read_csv(W / JOBS[1][0] / "oof.csv")] + ([pd.read_csv(SCREEN / "oof.csv")] if SCREEN else [])
oof = parts[0].copy()
for l in LAB:
    for o in parts[1:]: oof[f"p_{l}"] = oof[f"p_{l}"].fillna(o[f"p_{l}"])
for o in parts[1:]:
    assert (o.StudyInstanceUID == oof.StudyInstanceUID).all() and (o.fold == oof.fold).all(), "oof parts disagree on studies/folds"
assert oof[[f"p_{l}" for l in LAB]].notna().all().all(), "some studies have no OOF prediction"
oof.to_csv(W / f"oof_{TAG}.csv", index=False); print("saved", f"oof_{TAG}.csv", oof.shape)

['fold0_v6cnxT20.pt', 'fold1_v6cnxT20.pt', 'fold2_v6cnxT20.pt', 'fold3_v6cnxT20.pt', 'fold4_v6cnxT20.pt']
saved oof_v6cnxT20.csv (4407, 40)


In [4]:
# ---- pooled 5-fold comparison on the FIXED old labels (confident), ranks within fold, vs the ConvNeXt-T 12-epoch runs ----
from scipy.stats import rankdata
KEY4 = LAB[:4]
want = {"baseline": "oof_all.csv", "convnext": "oof_v6convnext.csv", "convnext_s43": "oof_v6convnext2.csv"}
paths = {}
for d, dirs, fs in os.walk(ROOT):
    dirs[:] = [x for x in dirs if x not in SKIP]
    for f in fs:
        for k, v in want.items():
            if f == v: paths.setdefault(k, []).append(Path(d) / f)
for k, v in paths.items():
    assert len(v) == 1, f"{k}: several files {v}"
assert "convnext" in paths, "attach the ConvNeXt 5-fold output (oof_v6convnext.csv)"
ids = oof.StudyInstanceUID.tolist(); fold = oof.fold.values.astype(int)
def get(df):
    d = df.set_index("StudyInstanceUID").loc[ids]
    return np.stack([d[f"p_{l}"].values for l in LAB], 1).astype(float), np.stack([d[f"y_{l}"].values for l in LAB], 1).astype(float)
P, Y = {}, None
for k, v in paths.items(): P[k], y = get(pd.read_csv(v[0])); Y = y if Y is None else Y
P[TAG], y = get(oof); assert np.allclose(y, Y, atol=1e-6), "y_ differs from the fixed old labels"
def frank(p):
    r = np.zeros_like(p)
    for k in np.unique(fold): m = fold == k; r[m] = np.apply_along_axis(rankdata, 0, p[m]) / m.sum()
    return r
RK = {n: frank(p) for n, p in P.items()}
def _auc(yb, s):
    n1 = int(yb.sum()); n0 = len(yb) - n1
    if n1 < 5 or n0 < 5: return np.nan
    r = rankdata(s); return (r[yb == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def aucs(R, rows=None):
    out = np.full(12, np.nan)
    for j in range(12):
        y = Y[:, j] if rows is None else Y[rows, j]; s = R[:, j] if rows is None else R[rows, j]
        m = (y >= .9) | (y <= .1)
        if m.sum() > 20: out[j] = _auc((y[m] >= .9).astype(int), s[m])
    return out
macro = lambda a: np.nanmean(a); key4 = lambda a: np.nanmean(a[:4])
cols = dict(RK)
if "convnext_s43" in RK: cols["ens convnext x2"] = (RK["convnext"] + RK["convnext_s43"]) / 2; cols[f"ens convnext x2 + {TAG}"] = (RK["convnext"] + RK["convnext_s43"] + RK[TAG]) / 3
cols[f"ens convnext + {TAG}"] = (RK["convnext"] + RK[TAG]) / 2
T = pd.DataFrame({n: aucs(r) for n, r in cols.items()}, index=LAB); T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("All 5 folds pooled (ranks within fold):\n", T.round(3).to_string())
print("\nper-fold macro:")
for k in range(5):
    r = np.where(fold == k)[0]; f = lambda R: macro(aucs(R, r))
    print(f"  fold {k}: convnext {f(RK['convnext']):.4f} | {TAG} {f(RK[TAG]):.4f} | delta {f(RK[TAG])-f(RK['convnext']):+.4f}")
rng = np.random.default_rng(0); d = []
for _ in range(300):
    b = rng.integers(0, len(ids), len(ids)); d.append(macro(aucs(RK[TAG], b)) - macro(aucs(RK["convnext"], b)))
dm = macro(aucs(RK[TAG])) - macro(aucs(RK["convnext"]))
print(f"\n{TAG} - convnext (12 ep seed 42): macro {dm:+.4f}  90% CI [{np.percentile(d,5):+.4f}, {np.percentile(d,95):+.4f}] (study resampling only)")
print("Pooled over 5 folds this is a much tighter test than fold 0 alone. If delta is >= +0.005 with at least 4/5 folds better, use it.")

All 5 folds pooled (ranks within fold):
                   baseline  convnext  v6cnxT20  ens convnext + v6cnxT20
ACL                  0.781     0.853     0.860                    0.861
MCL                  0.725     0.797     0.802                    0.806
Medial Meniscus      0.770     0.881     0.900                    0.896
Lateral Meniscus     0.736     0.786     0.827                    0.816
Medial OA            0.826     0.860     0.862                    0.863
Lateral OA           0.747     0.777     0.788                    0.788
PF OA                0.717     0.727     0.734                    0.734
Effusion             0.822     0.840     0.841                    0.843
Synovitis            0.772     0.791     0.782                    0.789
Baker's              0.756     0.850     0.851                    0.856
Contusion            0.755     0.824     0.827                    0.830
Fracture             0.780     0.851     0.859                    0.859
MACRO                0.